# Mapping Fertility Services in the United Kingdom
### DATASCI 530 - Project 1: Web scraping

**Research question:** What fertility facilities appear in the Human Fertilisation and Embryology Authority (HFEA) directory, where are they located, and what treatment outcome statistics are available on their profiles?

**Website:** https://www.hfea.gov.uk/choose-a-fertility-clinic/search/

**Data collection date:** October 19-21, 2026 (I had to run the scraping process twice due to corrections)

**Data Collected and Scope:** I collected all clinic listings available through the HFEA's A–Z directory, without restricting the sample to a particular region or type of clinic. For each listing, I collected clinic names, addresses, postcodes, external website links, and HFEA profile links where available. For clinics with individual profiles, I focused on three treatment outcome measures: IVF births per embryo transferred, IUI pregnancies per treatment, and donor insemination births per treatment. I collected these statistics for three age groups (under 38, 38 and over, and all ages), along with national comparison rates and reporting periods where available. I also used the public Postcodes.io API to obtain geographic information about the clinics. Not every listed facility offers these treatments or publishes outcome statistics.

**Access and ethics:** Before collecting data, I contacted the HFEA to ask about using publicly available information from its website for a noncommercial academic project. They indicated that my proposed educational use would likely fall within applicable copyright exceptions. I limited the scraping process to publicly accessible pages, used delays between requests, and did not attempt to access restricted information.

### Background information

The Human Fertilisation and Embryology Authority (HFEA) has an online directory of fertility clinics in the United Kingdom. The directory is organized alphabetically (A–Z). Each listing includes the clinic's name and address, and some also provide a link to the clinic's own website.

Some clinics also have individual profiles hosted on the HFEA website, while others are only listed in the directory. These profiles provide additional information about the clinics, including, where available, fertility treatment outcome statistics. The statistics include outcomes such as IVF births per embryo transferred, IUI pregnancies per treatment, and donor insemination births per treatment. Results are presented for different age groups (under 38, 38 and over, and all ages), alongside national comparison rates.

This structure makes the HFEA website suitable for web scraping because clinic information is distributed across multiple alphabetical directory pages, while more detailed information must be accessed by navigating to individual clinic profiles.

### Project Aim

In this project, I aim to collect information about fertility clinics in the UK using web scraping. First, I will scrape the HFEA's A–Z clinic directory to collect clinic names, addresses, postcodes, and website links. I will then visit individual clinic profiles, where available, to collect statistics on fertility treatment outcomes for different age groups.

I also aim to use clinic postcodes to identify their geographic locations. Finally, I will combine the information into a single dataset and use descriptive statistics to explore the geographic distribution of fertility clinics in the UK and the availability of treatment outcome data (continued in Project 2).

## 1. Set up

I use Selenium to navigate pages, pandas to organize records, requests for the postcode API, and regular expressions to identify UK postcodes.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re
import string
import time

import numpy as np
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

Path("data").mkdir(exist_ok=True)
collection_started = datetime.now(timezone.utc)
print("Collection started (UTC):", collection_started.isoformat())

driver = webdriver.Chrome()


Collection started (UTC): 2026-10-21T22:36:50.173555+00:00


## 2. Scrape the full A–Z directory

I visit all 26 alphabetical directory pages using this for loop. For each clinic listing, I locate the HTML container and extract the clinic name, address, postcode, profile link, and website. The directory navigation uses a five-second pause per page.

In [4]:
def extract_postcode(address_parts):
    address_text = " ".join(address_parts)
    pattern = r"\b[A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2}\b"
    match = re.search(pattern, address_text, flags=re.IGNORECASE)
    return match.group().upper() if match else None

all_clinics = []

for letter in string.ascii_lowercase:
    url = f"https://www.hfea.gov.uk/choose-a-fertility-clinic/search/all-clinics/?alpha={letter}"
    driver.get(url)
    time.sleep(5)

    clinic_containers = driver.find_elements(
        By.XPATH,
        "//div[contains(@class, 'row')][./div[contains(@class, 'col-md-12')]/h2[contains(@class, 'h3')]]"
    )
    print(f"{letter.upper()}: {len(clinic_containers)} listings")

    for container in clinic_containers:
        name = container.find_element(By.XPATH, "./div/h2").text
        profile_links = container.find_elements(By.XPATH, "./div/h2/a")
        profile_url = profile_links[0].get_attribute("href") if profile_links else None
        address_elements = container.find_elements(
            By.XPATH, "./div[contains(@class, 'col-md-6')][1]//li"
        )
        address_parts = [element.text.strip() for element in address_elements if element.text.strip()]
        website_links = container.find_elements(By.XPATH, ".//a[contains(@title, 'Website link')]")

        all_clinics.append({
            "clinic_name": name,
            "address": ", ".join(address_parts),
            "postcode": extract_postcode(address_parts),
            "profile_url": profile_url,
            "directory_url": url,
            "website": website_links[0].get_attribute("href") if website_links else None
        })

all_clinics_df = pd.DataFrame(all_clinics)
all_clinics_df.to_csv("data/hfea_clinics_raw.csv", index=False)
print("Total directory listings:", len(all_clinics_df))
display(all_clinics_df.head())

A: 18 listings
B: 12 listings
C: 47 listings
D: 5 listings
E: 3 listings
F: 13 listings
G: 5 listings
H: 7 listings
I: 7 listings
J: 6 listings
K: 3 listings
L: 29 listings
M: 4 listings
N: 5 listings
O: 4 listings
P: 5 listings
Q: 0 listings
R: 6 listings
S: 12 listings
T: 8 listings
U: 2 listings
V: 1 listings
W: 11 listings
X: 1 listings
Y: 1 listings
Z: 0 listings
Total directory listings: 215


,clinic_name,address,postcode,profile_url,directory_url,website
0,Abbey Fertility Clinic,"6 Telford Court, Chestergates Business Park, C...",CH1 6LT,None,https://www.hfea.gov.uk/choose-a-fertility-cli...,None
1,Aberdeen Fertility Centre,"Department of Obstetrics & Gynaecology, Aberde...",AB25 2ZL,https://www.hfea.gov.uk/choose-a-fertility-cli...,https://www.hfea.gov.uk/choose-a-fertility-cli...,http://www.aberdeenfertility.org.uk/
2,Acorn Fertility,"112a, Cavendish Road, Colliers Wood, Merton, S...",SW19 2EZ,https://www.hfea.gov.uk/choose-a-fertility-cli...,https://www.hfea.gov.uk/choose-a-fertility-cli...,None
3,Adrian Lower,"9, Upper Wimpole Street, London, W1G 6LJ",W1G 6LJ,None,https://www.hfea.gov.uk/choose-a-fertility-cli...,None
4,Agora Clinic Brighton,"138, Dyke Road, Brighton, BN1 5PA",BN1 5PA,https://www.hfea.gov.uk/choose-a-fertility-cli...,https://www.hfea.gov.uk/choose-a-fertility-cli...,http://www.agoraclinic.co.uk/


## 3. Clean clinic listings and inspect missing values

I preserve the raw CSV, remove duplicate clinic name/postcode combinations, and document one observed postcode transcription/character issue. Corrections are not inferred automatically.

In [ ]:
clinics_clean_df = all_clinics_df.drop_duplicates(
    subset=["clinic_name", "postcode"]
).copy().reset_index(drop=True)

# Correct the postcode for TFP Wessex Fertility after checking
# the original directory listing
clinics_clean_df.loc[
    clinics_clean_df["clinic_name"] == "TFP Wessex Fertility", "postcode"
] = "SO30 2UN"

print("Raw listings:", len(all_clinics_df))
print("Unique clinic listings:", len(clinics_clean_df))
print("Missing values by field:")
display(clinics_clean_df.isna().sum().to_frame("missing"))
clinics_clean_df.to_csv("data/hfea_clinics_clean.csv", index=False)

Raw listings: 215
Unique clinic listings: 212
Missing values by field:


,missing
clinic_name,0
address,0
postcode,0
profile_url,92
directory_url,0
website,124


## 4. Add geographic information using Postcodes.io

The public API returns the local authority, local-authority code, country, and coordinates for valid postcodes. I submit postcodes in batches of up to 100 and leave unsuccessful lookups missing. A historic Canterbury postcode is documented separately.

In [6]:
clinics_clean_df["postcode"] = (
    clinics_clean_df["postcode"].astype("string").str.upper().str.strip()
)
postcodes = clinics_clean_df["postcode"].dropna().unique().tolist()
geo_data = []

for start in range(0, len(postcodes), 100):
    batch = postcodes[start:start + 100]
    response = requests.post(
        "https://api.postcodes.io/postcodes", json={"postcodes": batch}, timeout=30
    )
    response.raise_for_status()
    for item in response.json()["result"]:
        result = item["result"]
        geo_data.append({
            "postcode": item["query"],
            "local_authority": result["admin_district"] if result else None,
            "local_authority_code": result["codes"]["admin_district"] if result else None,
            "country": result["country"] if result else None,
            "latitude": result["latitude"] if result else None,
            "longitude": result["longitude"] if result else None
        })
    time.sleep(2)

geo_df = pd.DataFrame(geo_data)
clinics_geo_df = clinics_clean_df.merge(geo_df, on="postcode", how="left", validate="many_to_one")
clinics_geo_df["coordinate_source"] = np.where(
    clinics_geo_df["latitude"].notna(), "active_postcode", None
)

# Historical postcode coordinates, recorded separately from current postcode matches.
mask = clinics_geo_df["postcode"] == "CT2 8BL"
clinics_geo_df.loc[mask, "latitude"] = 51.282289
clinics_geo_df.loc[mask, "longitude"] = 1.073795
clinics_geo_df.loc[mask, "coordinate_source"] = "terminated_postcode"
clinics_geo_df.loc[mask, "country"] = "England"
clinics_geo_df["current_location_verified"] = clinics_geo_df["coordinate_source"] == "active_postcode"

clinics_geo_df["postcode_issue"] = None
clinics_geo_df.loc[clinics_geo_df["postcode"] == "S4A 9HJ", "postcode_issue"] = "Unmatched postcode; verify manually"
clinics_geo_df.loc[clinics_geo_df["postcode"] == "JR2 3RA", "postcode_issue"] = "Unusual Jersey postcode; verify manually"
clinics_geo_df.loc[mask, "postcode_issue"] = "Terminated postcode; historical coordinates only"

clinics_geo_df.to_csv("data/hfea_clinics_geographic.csv", index=False)
print("Clinics with coordinates:", clinics_geo_df["latitude"].notna().sum())
display(clinics_geo_df.loc[clinics_geo_df["postcode_issue"].notna(),
    ["clinic_name", "postcode", "postcode_issue"]])

Clinics with coordinates: 207


,clinic_name,postcode,postcode_issue
51,"Centre for Reproduction & Gynaecology Wales, S...",S4A 9HJ,Unmatched postcode; verify manually
118,Jersey Women’s Health Limited,JR2 3RA,Unusual Jersey postcode; verify manually
138,London Women's Clinic - Canterbury,CT2 8BL,Terminated postcode; historical coordinates only


## 5. Scrape treatment outcomes from individual clinic profiles

For each clinic with a profile URL, I navigate to its page and find the `birth-statistics` section. Some clinics don't have birth statistics. I record those as missing. The site uses radio buttons to change age groups, so I click them and read the updated HTML. I extract **three outcomes (IVF births per embryo transferred, IUI pregnancies per treatment, and donor insemination births per treatment) × three age groups (under 38, 38 and over, and all ages)** for each clinic that has birth statistics. The script pauses five seconds between profiles and saves progress after each visit.

In [15]:
outcome_groups = {
    "ivfembryo": "IVF births per embryo transferred",
    "iuibirths": "IUI pregnancies per treatment",
    "dibirths": "Donor insemination births per treatment"
}

age_groups = ["Under 38", "38 and over", "All"]


def extract_percentage(container, selector):
    elements = container.find_elements(By.CSS_SELECTOR, selector)

    if not elements:
        return np.nan

    value = elements[0].get_attribute("textContent").strip()

    if not re.fullmatch(r"\d+(?:\.\d+)?%", value):
        return np.nan

    return float(value.replace("%", ""))

In [16]:

def scrape_clinic_statistics(driver, clinic_name, profile_url, directory_url):

    # Return to the alphabetical directory page
    driver.get(directory_url)
    time.sleep(5)

    # Find the clinic's link on the directory page
    clinic_links = driver.find_elements(
        By.XPATH,
        "//h2[contains(@class, 'h3')]/a"
    )

    clicked = False

    for link in clinic_links:
        href = link.get_attribute("href")

        if href and href.rstrip("/") == profile_url.rstrip("/"):
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                link
            )
            time.sleep(1)

            # JavaScript click avoids the intercepted-click error
            driver.execute_script("arguments[0].click();", link)

            clicked = True
            break

    if not clicked:
        return [], "profile_link_not_found"

    # Wait until the profile page loads
    WebDriverWait(driver, 20).until(
        lambda d: d.current_url.rstrip("/") == profile_url.rstrip("/")
    )

    time.sleep(2)

    if not driver.find_elements(By.ID, "birth-statistics"):
        return [], "no_statistics_section"

    results = []

    for chart_index, (group, outcome) in enumerate(outcome_groups.items()):
        for i, age_group in enumerate(age_groups):

            buttons = driver.find_elements(
                By.CSS_SELECTOR,
                f"#birth-statistics input[name='{group}']"
            )

            if len(buttons) != 3:
                raise ValueError(
                    f"Expected three age buttons for {group}; "
                    f"found {len(buttons)}"
                )

            driver.execute_script(
                "arguments[0].click();", buttons[i]
            )
            time.sleep(0.5)

            charts = driver.find_elements(
                By.CSS_SELECTOR,
                "#birth-statistics .statistics-container"
            )

            if len(charts) <= chart_index:
                raise ValueError(
                    f"Missing chart {chart_index} for {outcome}"
                )

            chart = charts[chart_index]

            results.append({
                "clinic_name": clinic_name,
                "profile_url": profile_url,
                "outcome": outcome,
                "age_group": age_group,
                "clinic_rate": extract_percentage(
                    chart, ".mean span"
                ),
                "national_rate": extract_percentage(
                    chart, ".nat-ave span"
                )
            })

    return results, "statistics_extracted"

In [18]:
profiles = (
    clinics_geo_df.dropna(subset=["profile_url"])
    .loc[lambda df: df["profile_url"].str.strip().ne("")]
    .drop_duplicates(subset="profile_url")
)
print("Profiles to visit:", len(profiles))

all_statistics = []
profile_log = []
for i, row in enumerate(profiles.itertuples(index=False), start=1):
    try:
        records, status = scrape_clinic_statistics(driver, row.clinic_name, row.profile_url, row.directory_url)
        all_statistics.extend(records)
        profile_log.append({"clinic_name": row.clinic_name, "profile_url": row.profile_url,
                            "status": status, "detail": ""})
        print(f"{i}/{len(profiles)} {row.clinic_name}: {status}, {len(records)} rows")
    except Exception as exc:
        profile_log.append({"clinic_name": row.clinic_name, "profile_url": row.profile_url,
                            "status": "error", "detail": str(exc)})
        print(f"{i}/{len(profiles)} {row.clinic_name}: ERROR: {exc}")

    pd.DataFrame(all_statistics, columns=[
        "clinic_name", "profile_url", "outcome", "age_group", "clinic_rate", "national_rate"
    ]).to_csv("data/hfea_clinic_statistics.csv", index=False)
    pd.DataFrame(profile_log).to_csv("data/hfea_profile_scrape_log.csv", index=False)
    time.sleep(5)

clinic_stats_df = pd.DataFrame(all_statistics, columns=[
    "clinic_name", "profile_url", "outcome", "age_group", "clinic_rate", "national_rate"
])
profile_log_df = pd.DataFrame(profile_log)
print("Outcome observations:", len(clinic_stats_df))
display(profile_log_df["status"].value_counts(dropna=False).to_frame("profiles"))

Profiles to visit: 120


1/120 Aberdeen Fertility Centre: statistics_extracted, 9 rows
2/120 Acorn Fertility: statistics_extracted, 9 rows
3/120 Agora Clinic Brighton: statistics_extracted, 9 rows
4/120 Agora Clinic Eastbourne: no_statistics_section, 0 rows
5/120 Andrology Solutions: statistics_extracted, 9 rows
6/120 Andrology Unit, Hammersmith Hospital: no_statistics_section, 0 rows
7/120 Aria Fertility: statistics_extracted, 9 rows
8/120 Assisted Reproduction and Gynaecology Centre: statistics_extracted, 9 rows
9/120 Assisted Reproduction Unit (ARU), University Hospital of Hartlepool: no_statistics_section, 0 rows
10/120 Avenues: statistics_extracted, 9 rows
11/120 Ayrshire Fertility Unit, University Hospital Crosshouse: no_statistics_section, 0 rows
12/120 Barts Health Centre for Reproductive Medicine: statistics_extracted, 9 rows
13/120 Beginnings at Epsom & St Helier NHS University Trust: statistics_extracted, 9 rows
14/120 Birmingham Women's Hospital: statistics_extracted, 9 rows
15/120 Bourn Hall Clini

,profiles
status,
statistics_extracted,96
no_statistics_section,22
profile_link_not_found,2


## 6. Extract reporting periods from profiles with statistics

Reported outcomes may cover different years. I revisit only the profiles that yielded outcome rows and read the reporting periods shown alongside the three charts. This is a second automated loop, with an eight-second delay between page visits.

In [19]:
def extract_reporting_periods(driver, clinic_name, profile_url):
    section = driver.find_element(By.ID, "birth-statistics")
    elements = section.find_elements(By.CSS_SELECTOR, "span.period")
    outcomes = list(outcome_groups.values())
    if len(elements) != 3:
        raise ValueError(f"Expected three reporting periods, found {len(elements)}")

    records = []
    for outcome, element in zip(outcomes, elements):
        dates = re.findall(r"\d{2}/\d{2}/\d{4}", element.get_attribute("textContent"))
        if len(dates) != 2:
            raise ValueError(f"Could not parse reporting dates for {outcome}")
        records.append({
            "profile_url": profile_url, "outcome": outcome,
            "period_start": pd.to_datetime(dates[0], format="%d/%m/%Y"),
            "period_end": pd.to_datetime(dates[1], format="%d/%m/%Y")
        })
    return records

successful_profiles = clinic_stats_df[["clinic_name", "profile_url"]].drop_duplicates()
period_records = []
period_log = []
for i, row in enumerate(successful_profiles.itertuples(index=False), start=1):
    try:
        driver.get(row.profile_url)
        WebDriverWait(driver, 20).until(lambda d: d.find_elements(By.ID, "birth-statistics"))
        period_records.extend(extract_reporting_periods(driver, row.clinic_name, row.profile_url))
        period_log.append({"clinic_name": row.clinic_name, "status": "extracted", "detail": ""})
    except Exception as exc:
        period_log.append({"clinic_name": row.clinic_name, "status": "error", "detail": str(exc)})
        print("Reporting period error:", row.clinic_name, exc)
    pd.DataFrame(period_records, columns=["profile_url", "outcome", "period_start", "period_end"]).to_csv(
        "data/hfea_reporting_periods.csv", index=False
    )
    pd.DataFrame(period_log).to_csv("data/hfea_reporting_period_log.csv", index=False)
    time.sleep(8)

periods_df = pd.DataFrame(period_records, columns=["profile_url", "outcome", "period_start", "period_end"])
print("Reporting period records:", len(periods_df))

Reporting period records: 288


## 7. Close the browser and combine the newly scraped data

The combined dataset keeps clinics without HFEA profile links and clinics without published outcome rates. I keep the missing values remain missing; I do not convert them into zero.

In [20]:
driver.quit()
collection_finished = datetime.now(timezone.utc)
print("Collection finished (UTC):", collection_finished.isoformat())
print("Elapsed:", collection_finished - collection_started)

if not clinic_stats_df.empty:
    clinic_stats_final_df = clinic_stats_df.merge(
        periods_df, on=["profile_url", "outcome"], how="left", validate="many_to_one"
    )
    clinic_stats_final_df["reporting_year"] = pd.to_datetime(
        clinic_stats_final_df["period_end"]
    ).dt.year
else:
    clinic_stats_final_df = clinic_stats_df.copy()
    for col in ["period_start", "period_end", "reporting_year"]:
        clinic_stats_final_df[col] = pd.NA

with_profiles = clinics_geo_df[clinics_geo_df["profile_url"].notna()].copy()
without_profiles = clinics_geo_df[clinics_geo_df["profile_url"].isna()].copy()
merged_profiles = with_profiles.merge(
    clinic_stats_final_df.drop(columns=["clinic_name"], errors="ignore"),
    on="profile_url", how="left", validate="one_to_many"
)
clinics_combined_df = pd.concat([merged_profiles, without_profiles], ignore_index=True)
clinics_combined_df.to_csv("data/hfea_clinics_combined.csv", index=False)
print("Combined rows:", len(clinics_combined_df))
print("Distinct facilities:", clinics_combined_df[["clinic_name", "postcode"]].drop_duplicates().shape[0])

Collection finished (UTC): 2026-10-22T00:02:15.370885+00:00
Elapsed: 1:25:25.197330
Combined rows: 980
Distinct facilities: 212


## 8. Quality checks and results

Lastly, I summarize the data collected from the HFEA website and perform some basic quality checks. First, I calculate the total number of unique fertility clinics and fertility outcome observations. I also check for missing clinic information, including addresses, postcodes, website links, clinic profiles, and geographic coordinates.

For fertility treatment outcomes, I calculate the number of observations, missing values, mean, and variance of clinic-level outcome percentages for each treatment type and age group.

Finally, I check for duplicate clinic listings and fertility outcome observations, as well as outcome percentages outside the possible range of 0–100%. I also summarize the results of the profile scraping process to see how many profiles were successfully scraped and whether any errors occurred or statistics were unavailable.

In [22]:

# Use the final combined dataset
final_df = clinics_combined_df.copy()

# Number of observations
clinic_level_df = final_df.drop_duplicates(
    subset=["clinic_name", "postcode"]
).copy()

outcome_df = final_df[final_df["outcome"].notna()].copy()

print("Total rows in final dataset:", len(final_df))
print("Unique fertility clinics:", len(clinic_level_df))
print("Fertility outcome observations:", len(outcome_df))

# Missing clinic-level information
print("\nMissing values in clinic information:")
display(
    clinic_level_df[[
        "clinic_name",
        "address",
        "postcode",
        "website",
        "profile_url",
        "country",
        "local_authority_code",
        "latitude",
        "longitude"
    ]].isna().sum().to_frame("missing_values")
)

# Fertility outcome descriptive statistics
print("\nFertility outcome descriptive statistics:")

outcome_summary = (
    outcome_df
    .groupby(["outcome", "age_group"])
    .agg(
        observations=("clinic_rate", "size"),
        missing_values=("clinic_rate", lambda x: x.isna().sum()),
        mean=("clinic_rate", "mean"),
        variance=("clinic_rate", "var")
    )
    .reset_index()
)

display(outcome_summary)

# Data quality checks
print("\nDuplicate clinic/outcome/age observations:")
print(
    outcome_df.duplicated(
        subset=["profile_url", "outcome", "age_group"]
    ).sum()
)

print("\nOutcome percentages outside 0–100%:")
print(
    (~outcome_df["clinic_rate"].dropna().between(0, 100)).sum()
)

print("\nOutcome observations missing reporting periods:")
print(
    outcome_df["period_start"].isna().sum()
)

print("\nOutcome observations missing reporting period end dates:")
print(
    outcome_df["period_end"].isna().sum()
)

print("\nInvalid reporting periods (end date before start date):")
print(
    (
        pd.to_datetime(outcome_df["period_end"])
        < pd.to_datetime(outcome_df["period_start"])
    ).sum()
)

# Scraping process quality checks
print("\nProfile scraping status:")
display(
    profile_log_df["status"]
    .value_counts(dropna=False)
    .to_frame("profiles")
)


Total rows in final dataset: 980
Unique fertility clinics: 212
Fertility outcome observations: 864

Missing values in clinic information:


,missing_values
clinic_name,0
address,0
postcode,0
website,124
profile_url,92
country,2
local_authority_code,3
latitude,5
longitude,5



Fertility outcome descriptive statistics:


,outcome,age_group,observations,missing_values,mean,variance
0,Donor insemination births per treatment,38 and over,96,21,9.413333,88.732252
1,Donor insemination births per treatment,All,96,19,14.649351,55.204375
2,Donor insemination births per treatment,Under 38,96,24,17.611111,83.395931
3,IUI pregnancies per treatment,38 and over,96,45,9.549020,121.772549
4,IUI pregnancies per treatment,All,96,40,12.446429,104.506169
5,IUI pregnancies per treatment,Under 38,96,48,15.291667,147.062057
6,IVF births per embryo transferred,38 and over,96,8,16.227273,36.062696
7,IVF births per embryo transferred,All,96,7,28.224719,49.880746
8,IVF births per embryo transferred,Under 38,96,8,33.772727,64.338558



Duplicate clinic/outcome/age observations:
0

Outcome percentages outside 0–100%:
0

Outcome observations missing reporting periods:
0

Outcome observations missing reporting period end dates:
0

Invalid reporting periods (end date before start date):
0

Profile scraping status:


,profiles
status,
statistics_extracted,96
no_statistics_section,22
profile_link_not_found,2


## 9. Discussion and limitations

The scraping algorithm automatically goes through the 26 alphabetical pages of the HFEA clinic directory and collects information from multiple clinic listings. It then navigates to individual clinic profiles by clicking their links in the directory and extracts available fertility treatment outcome statistics for different age groups. I also use clinic postcodes to collect geographic information. The resulting datasets are saved as CSV files.

I collected information on 212 unique fertility clinics from the HFEA directory. Of these, 120 had individual HFEA profile links. I was able to extract treatment outcome statistics from 96 profiles. This resulted in a total of 864 observations across three treatment outcomes and three age groups.

Not all the information I wanted to collect was available. Clinic names, addresses, and postcodes were available for all 212 clinics, but 124 did not have external website links and 92 did not have HFEA profile links. I was also unable to obtain geographic coordinates for five clinics. Among the 120 profiles, 22 did not have a birth statistics section. For another two clinics, the scraping algorithm could not locate their profile links during automated navigation. Unlike clinics without a statistics section, I cannot determine whether treatment outcome statistics were available for these two clinics, so I treat them as unsuccessful scraping attempts.

There were also differences in the availability of treatment outcome statistics. Missing values were particularly common for IUI pregnancies per treatment, with 40 to 48 missing values depending on the age group. In comparison, IVF births per embryo transferred had only 7 to 8 missing values per age group. This suggests that although the HFEA directory provides a useful source of information about fertility clinics, the availability of treatment outcome data varies considerably across treatments.

The quality checks identified three duplicate clinic listings in the raw data, which were removed during cleaning. I did another quality check and thankfully did not find any duplicate fertility outcome observations or outcome percentages outside the possible range of 0–100%.

One important limitation is that the facilities listed in the directory do not necessarily offer the same services, and some may not provide fertility treatments directly. Additionally, the outcome measures use different denominators and may cover different reporting periods, meaning that they should not be compared directly. Missing outcome percentages may reflect unavailable or suppressed statistics rather than unsuccessful treatments, so I kept these values as missing instead of replacing them with zero.

Finally, the data reflect the information available on the HFEA website at the time of collection (October 19–21, 2026). Changes to clinic listings, published statistics, or the website's structure could affect the results of future scraping attempts.